In [ ]:
# fl_pure_non_iid_with_attacks_cifar10.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from collections import defaultdict

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

NUM_CLIENTS = 10
NUM_ROUNDS = 500
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack parameters
MALICIOUS_PER_ROUND = 3
LABEL_FLIP_RATIO = 0.3  # Only flip a portion of data

# Combined scaling + noise attack (previously separate)
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Backdoor params (adjusted for CIFAR-10's 3 channels)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

# Random Gaussian Noise Attack (previously Model Replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# DCT config
NUM_DCT_COEFFS_SAVE = 300
OUTPUT_CSV = "3finaltrial_cifar10.csv"  # Updated filename

# ---------------------------
# Flexible CNN Model for CIFAR-10
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=3, input_size=(32, 32), num_classes=10):
        """
        Flexible CNN that adapts to different input sizes and channels

        Args:
            input_channels: Number of input channels (1 for grayscale, 3 for RGB)
            input_size: Tuple of (height, width)
            num_classes: Number of output classes
        """
        super(SimpleCNN, self).__init__()
        self.input_channels = input_channels
        self.input_size = input_size

        # First convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Third convolutional layer (added for better feature extraction in color images)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)

        # Calculate the size after three pooling operations
        h, w = input_size
        h = h // 8  # After three pool operations (each halves the size)
        w = w // 8

        # Calculate flattened size
        self.flattened_size = 128 * h * w

        # Fully connected layers
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        print(f"Model configured for CIFAR-10: {input_channels} channels, size {input_size}")
        print(f"Flattened size after conv layers: {self.flattened_size}")

    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = x.view(-1, self.flattened_size)

        # Fully connected layers with dropout
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)

        return x

# ---------------------------
# Pure Non-IID Split (One Class Per Client)
# ---------------------------
def pure_non_iid_split(dataset, num_clients):
    """Each client gets data from exactly one class"""
    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}

    # Assign one unique class to each client
    classes = list(range(10))  # 10 classes for CIFAR-10
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % 10]
        indices = label_to_indices[class_id]

        # Take ~500 samples per client (balanced for CIFAR-10)
        samples_per_client = min(500, len(indices))
        selected_indices = random.sample(indices, samples_per_client)
        client_indices[client_id].extend(selected_indices)

        # Remove used indices to avoid overlap
        for idx in selected_indices:
            label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature Extraction
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE, reference_vec=None):
    """
    Compute absolute DCT coefficients for energy-based features, plus
    some sign-aware / reference-aware features (pos_frac, neg_frac, cosine_with_ref).
    """
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # DCT coefficients
    coefs_save = coeffs[:num_coeffs]
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    # Statistical features
    feats["update_norm"] = float(np.linalg.norm(delta_vec))
    feats["update_mean"] = float(np.mean(delta_vec))
    feats["update_std"] = float(np.std(delta_vec))
    feats["update_skew"] = float(stats.skew(delta_vec))
    feats["update_kurtosis"] = float(stats.kurtosis(delta_vec))

    # Signed / sign-aware features
    pos_count = int(np.sum(delta_vec > 0))
    neg_count = int(np.sum(delta_vec < 0))
    feats["pos_frac"] = float(pos_count / max(1, delta_vec.size))
    feats["neg_frac"] = float(neg_count / max(1, delta_vec.size))
    feats["sign_ratio"] = float((pos_count - neg_count) / max(1, delta_vec.size))
    feats["signed_dct_mean"] = float(np.mean(signed_coeffs[:10]))

    # Cosine similarity w.r.t reference vector if provided
    if reference_vec is not None and np.linalg.norm(reference_vec) > 0 and np.linalg.norm(delta_vec) > 0:
        cos = float(np.dot(delta_vec, reference_vec) / (np.linalg.norm(delta_vec) * np.linalg.norm(reference_vec)))
        feats["cosine_with_ref"] = cos
    else:
        feats["cosine_with_ref"] = 0.0

    # Frequency bands
    bands = {"very_low": (0, 10), "low": (10, 50), "mid": (50, 150), "high": (150, 300)}
    total_energy = np.sum(coeffs) + 1e-12

    for band_name, (start, end) in bands.items():
        band_coeffs = coeffs[start:min(end, len(coeffs))]
        if len(band_coeffs) > 0:
            feats[f"{band_name}_energy"] = float(np.sum(band_coeffs))
            feats[f"{band_name}_ratio"] = float(np.sum(band_coeffs) / total_energy)
        else:
            feats[f"{band_name}_energy"] = 0.0
            feats[f"{band_name}_ratio"] = 0.0

    # Attack-specific features
    sorted_coeffs = np.sort(coeffs)[::-1]
    denom = (np.mean(sorted_coeffs[1:5]) + 1e-12) if sorted_coeffs.size > 4 else (np.mean(sorted_coeffs[1:]) + 1e-12)
    feats["spike_ratio"] = float(sorted_coeffs[0] / denom)
    feats["top5_energy_ratio"] = float(np.sum(sorted_coeffs[:5]) / total_energy)
    feats["spectral_entropy"] = float(-np.sum((coeffs/total_energy) * np.log(coeffs/total_energy + 1e-12)))

    return feats

# ---------------------------
# Attack Functions (adapted for CIFAR-10)
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes):
    """Flip LABEL_FLIP_RATIO portion of original-class labels to a random other class"""
    if not client_classes:
        return None

    original_class = client_classes[0]
    available_targets = [c for c in range(10) if c != original_class]
    target_class = random.choice(available_targets)

    xs, ys = [], []
    for img, lab in dataset_subset:
        # Only flip LABEL_FLIP_RATIO of the data
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)

    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes):
    """Backdoor: poisons BACKDOOR_POISON_RATIO of data with a tiny trigger"""
    if not client_classes:
        return None

    original_class = client_classes[0]
    available_targets = [c for c in range(10) if c != original_class]
    target_class = random.choice(available_targets)

    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)

    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()

            # Apply trigger to all 3 channels (or you can modify to specific channel)
            if trigger_pos == "bottom_right":
                arr[:, -BACKDOOR_TRIGGER_SIZE:, -BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[:, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start = 16 - BACKDOOR_TRIGGER_SIZE // 2  # Center for 32x32 images
                arr[:, start:start+BACKDOOR_TRIGGER_SIZE, start:start+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE

            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)

    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL Utilities
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

# ---------------------------
# Main Function
# ---------------------------
def main():
    print("🚀 Creating Pure Non-IID FL Dataset for CIFAR-10 with attacks mapping: 0-clean,1-labelflip,2-scale_and_noise,3-random_gaussian,4-backdoor,5-sparse_spike")

    # Setup CIFAR-10 dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Simple normalization
    ])

    cifar_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

    # Pure non-IID split
    client_indices = pure_non_iid_split(cifar_train, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, cifar_train)

    print("Client class distribution:")
    for cid, classes in client_classes.items():
        print(f"Client {cid}: classes {classes}, samples: {len(client_indices[cid])}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(cifar_train, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Initialize global model with flexible architecture for CIFAR-10
    global_model = SimpleCNN(
        input_channels=3,  # CIFAR-10 has 3 channels (RGB)
        input_size=(32, 32),  # CIFAR-10 images are 32x32
        num_classes=10
    ).to(DEVICE)
    global_state = copy.deepcopy(global_model.state_dict())

    rows = []
    stats = defaultdict(int)

    for rnd in range(NUM_ROUNDS):
        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models = {}

        # Train each client locally
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(global_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                client_class_list = client_classes[cid]

                if attack == 1:  # Label flip -> apply before training
                    attack_data = create_label_flip_attack([cifar_train[i] for i in indices], client_class_list)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant

                        xs_t = torch.stack([x for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = torch.utils.data.TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        # Fallback to clean training
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        loader = client_loaders_clean[cid]
                        local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    loader = client_loaders_clean[cid]
                    local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT use local data
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 4:  # Backdoor: poison a portion of local data before training
                    attack_data = create_backdoor_attack([cifar_train[i] for i in indices], client_class_list)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.stack([x for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = torch.utils.data.TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        loader = client_loaders_clean[cid]
                        local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    loader = client_loaders_clean[cid]
                    local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                loader = client_loaders_clean[cid]
                local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models[cid] = local_model

        # Compute deltas and apply attack-specific transforms
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(global_state, local_models[cid].state_dict())
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # compute reference vector (mean of clean deltas) for cosine similarity
        clean_deltas = [client_deltas[cid] for cid in range(NUM_CLIENTS) if client_label_gt[cid] == 0]
        if len(clean_deltas) > 0:
            mean_clean_delta = np.mean(np.stack(clean_deltas, axis=0), axis=0)
        else:
            mean_clean_delta = np.zeros_like(next(iter(client_deltas.values())))

        # Extract features and save to dataset
        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE, reference_vec=mean_clean_delta)

            row = {
                "client_id": int(cid),
                "round": int(rnd),
                "label": int(client_label_gt[cid]),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid])
            }
            row.update(feats)
            rows.append(row)
            stats[int(client_label_gt[cid])] += 1

        # FedAvg with clean clients only
        fed_clean_deltas = [client_deltas[cid] for cid in range(NUM_CLIENTS) if client_label_gt[cid] == 0]
        if len(fed_clean_deltas) > 0:
            mean_delta = np.mean(np.stack(fed_clean_deltas, axis=0), axis=0)
            new_state = unflatten_to_state(global_state, mean_delta)
            global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
            global_model.load_state_dict(global_state)

        if (rnd + 1) % 50 == 0:
            print(f"✅ Round {rnd+1}/{NUM_ROUNDS} completed. Samples collected: {len(rows)}")

    # Save dataset
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n🎉 Pure Non-IID dataset saved to: {OUTPUT_CSV}")
    print(f"📊 Final dataset shape: {df.shape}")
    print(f"📈 Class distribution: {dict(stats)}")

if __name__ == "__main__":
    main()

🚀 Creating Pure Non-IID FL Dataset for CIFAR-10 with attacks mapping: 0-clean,1-labelflip,2-scale_and_noise,3-random_gaussian,4-backdoor,5-sparse_spike


100%|██████████| 170M/170M [00:03<00:00, 48.8MB/s]


Client class distribution:
Client 0: classes [7], samples: 500
Client 1: classes [3], samples: 500
Client 2: classes [2], samples: 500
Client 3: classes [8], samples: 500
Client 4: classes [5], samples: 500
Client 5: classes [6], samples: 500
Client 6: classes [9], samples: 500
Client 7: classes [4], samples: 500
Client 8: classes [0], samples: 500
Client 9: classes [1], samples: 500
Model configured for CIFAR-10: 3 channels, size (32, 32)
Flattened size after conv layers: 2048
✅ Round 50/500 completed. Samples collected: 500
✅ Round 100/500 completed. Samples collected: 1000
✅ Round 150/500 completed. Samples collected: 1500
✅ Round 200/500 completed. Samples collected: 2000
✅ Round 250/500 completed. Samples collected: 2500
✅ Round 300/500 completed. Samples collected: 3000
✅ Round 350/500 completed. Samples collected: 3500
✅ Round 400/500 completed. Samples collected: 4000
✅ Round 450/500 completed. Samples collected: 4500
✅ Round 500/500 completed. Samples collected: 5000

🎉 Pure N